In [1]:
import os

import matplotlib.pyplot as plt

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

import numpy as onp
import numpy.typing as npt
import jax
import jax.numpy as jnp
from typing import List, Callable, Iterable, Tuple

from gridops_multidim import BSplineInterpolationAxis
from gridops_multidim import set_up_grid_axis
from gridops_multidim import create_anterpolation_operator
from gridops_multidim import create_restriction_operator, create_prolongation_operator, \
    create_interaction_operator
from gridops_multidim import create_compute_U_oneplus, create_compute_U_and_f_oneplus

import sys

sys.path.append("/home/florian/PhD/work/code/msm_for_nn/")

from msmfornn.splines.nesting import compute_J_zeroplus


# Basic settings

In [2]:
p = 6
order = p - 1

In [3]:
J_zeroplus = compute_J_zeroplus(p)
J = jnp.concatenate((J_zeroplus[::-1][:-1], J_zeroplus))

# Construct grids

In [4]:
length = 10.0
max_gridlevel = 4

grid_axes_all_levels = [None]  # there is no grid at level zero
for l in range(1, max_gridlevel + 1):
    h = length / (2 ** (max_gridlevel - l))
    print(l, h)
    grid_axis = set_up_grid_axis(length=length, h=h, p=p, J_zeroplus=J_zeroplus, periodic=False)
    grid_axes_all_levels.append(grid_axis)

1 1.25
2 2.5
3 5.0
4 10.0


# Create particle configuration

In [5]:
n_particles = 100
ndim = 3

rng = onp.random.default_rng(1632794)
xs = rng.uniform(0., length, size=(n_particles, ndim))
qs = rng.uniform(-1., 1., size=n_particles)

# Functions

In [6]:
def arbitrary_dim_outer(*xi: jax.Array) -> jax.Array:
    """Compute the outer product of an arbitrary number of arrays"""
    return jnp.prod(jnp.array(jnp.meshgrid(*xi, indexing="ij")), axis=0)

def make_multi_indices_one_particle(*inds_individual_axes):
    # TODO: name of this function and its arguments?
    multi_inds = jnp.array(
        [arr.ravel() for arr in jnp.meshgrid(*inds_individual_axes, indexing="ij")]
    ).T
    return multi_inds

def make_evaluate_bspline_basis_multidim_one_particle(grids):
    n_dim = len(grids)
    dims = tuple(g.n_total for g in grids)

    def evaluate_bspline_basis_multidim_one_particle(position):
        spline_outputs_one_particle = [
            grids[idx_cartesian].evaluate_bspline_basis_for_one_particle(
                position[idx_cartesian]
            )
            for idx_cartesian in range(n_dim)
        ]
        vals_individual_axes = [spl[0] for spl in spline_outputs_one_particle]
        inds_individual_axes = [spl[1] for spl in spline_outputs_one_particle]

        vals_flat = arbitrary_dim_outer(*vals_individual_axes).ravel()

        multi_inds = make_multi_indices_one_particle(*inds_individual_axes)
        inds_flat = jax.vmap(
            lambda mi: jnp.ravel_multi_index(mi, dims=dims, mode="clip")
        )(multi_inds)

        return vals_flat, inds_flat

    return evaluate_bspline_basis_multidim_one_particle

In [23]:
class BSplineInterpolationGrid:
    def __init__(self, axes: List[BSplineInterpolationAxis]):
        self.axes = axes
        
        self.shape = tuple(g.n_total for g in axes)
        self.ndim = len(axes)
        self.size = int(onp.prod(self.shape))
        
    def evaluate_bspline_basis_one_particle(self, position):
        spline_outputs_one_particle = [
            self.axes[idx_cartesian].evaluate_bspline_basis_for_one_particle(
                position[idx_cartesian]
            )
            for idx_cartesian in range(self.ndim)
        ]
        vals_individual_axes = [spl[0] for spl in spline_outputs_one_particle]
        inds_individual_axes = [spl[1] for spl in spline_outputs_one_particle]

        vals_flat = arbitrary_dim_outer(*vals_individual_axes).ravel()

        multi_inds = make_multi_indices_one_particle(*inds_individual_axes)
        # TODO: mode?
        inds_flat = jax.vmap(
            lambda mi: jnp.ravel_multi_index(mi, dims=self.shape, mode="clip")
        )(multi_inds)

        return vals_flat, inds_flat
    
    def evaluate_bspline_basis_multiparticle(self, positions):
        return jax.vmap(self.evaluate_bspline_basis_one_particle)(positions)
    
    def evaluate_bspline_basis_gradient_multiparticle(self, positions):
        return jax.vmap(
            jax.jacfwd(self.evaluate_bspline_basis_one_particle, has_aux=True)
        )(positions)
        

In [24]:
axes_level_one = (grid_axes_all_levels[1], ) * ndim
grid_level_one = BSplineInterpolationGrid(axes_level_one)

In [25]:
grid_level_one.shape

(15, 15, 15)

In [26]:
@jax.jit
def evaluate_bspline_basis_multiparticle(pos):
    return grid_level_one.evaluate_bspline_basis_multiparticle(pos)


@jax.jit
def evaluate_bspline_basis_gradient_multiparticle(pos):
    return grid_level_one.evaluate_bspline_basis_gradient_multiparticle(pos)

In [28]:
jax.device_put(xs)
%timeit evaluate_bspline_basis_multiparticle(xs)

236 µs ± 13.9 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [30]:
jax.device_put(xs)
%timeit evaluate_bspline_basis_gradient_multiparticle(xs)

429 µs ± 1.39 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [34]:
vals, inds = evaluate_bspline_basis_multiparticle(xs)
grads, _ = evaluate_bspline_basis_gradient_multiparticle(xs)

In [35]:
@jax.jit
def combined(pos):
    vals, inds = evaluate_bspline_basis_multiparticle(xs)
    grads, _ = evaluate_bspline_basis_gradient_multiparticle(xs)
    return inds, vals, grads

In [39]:
jax.device_put(xs)
%timeit combined(xs)

611 µs ± 741 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
